# CLIP4CAD-GFA v4.8.3 Training

**Fixed Cross-Modal Alignment Training**

## Key Fixes from v4.8.2
- **Bidirectional alignment**: Both text→geo and geo→text get gradients (no broken detach)
- **Text-anchored contrastive**: Text alignment gets 2x weight
- **Unified training**: All modalities train together from start (no BRep-PC pre-training that excludes text)
- **Lower tau=0.05**: Sharper contrastive signal
- **Proper metrics**: Track Text-BRep and Text-PC cosine separately

## Training Stages
| Stage | Epochs | LR | Goal |
|-------|--------|-----|------|
| 0 | 25 | 2e-4 → 1e-5 | Full 3-way alignment (Text + BRep + PC together) |
| 1 | 10 | 1e-5 | Hard negative refinement |

In [ ]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

# PC and Text files
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/gfa_v4_8_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS (v4.8.3 - Unified Training)
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 256

# Stage 0: Full 3-way alignment (all modalities together)
STAGE0_EPOCHS = 25
STAGE0_LR = 2e-4
STAGE0_WARMUP = 2

# Stage 1: Hard negative refinement
STAGE1_EPOCHS = 10
STAGE1_LR = 1e-5

TOTAL_EPOCHS = STAGE0_EPOCHS + STAGE1_EPOCHS

WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
MAX_TRAIN_SAMPLES = 111000

print("Configuration (v4.8.3 - Unified Training):")
print(f"  Data root: {DATA_ROOT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Stage 0: {STAGE0_EPOCHS} epochs, LR={STAGE0_LR} (all modalities together)")
print(f"  Stage 1: {STAGE1_EPOCHS} epochs, LR={STAGE1_LR} (hard negative refinement)")
print(f"  Total: {TOTAL_EPOCHS} epochs")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Import Dataset
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

print("Using GFAMappedDataset with use_autobrep=True")
print("v4.8.3: All modalities trained together from the start")

In [ ]:
# Cell 4: Load Datasets
gc.collect()
torch.cuda.empty_cache()

print("Loading datasets with AutoBrep topology...")
print("=" * 60)

# Train dataset - LOAD TO RAM
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset - ON DISK
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\n" + "=" * 60)
print("DATASETS READY!")

In [ ]:
# Cell 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

STEPS_PER_EPOCH = len(train_loader)
print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

In [ ]:
# Cell 6: Batch Remapping Function
def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        # Remove 'brep_' prefix for model compatibility
        if k.startswith('brep_'):
            new_key = k[5:]  # Remove 'brep_' prefix
            remapped[new_key] = v
        else:
            remapped[k] = v
    
    # Split pc_features into local and global
    if 'pc_features' in remapped:
        pc = remapped.pop('pc_features')
        remapped['pc_local_features'] = pc[:, :-1, :]   # (B, N, 1024)
        remapped['pc_global_features'] = pc[:, -1, :]   # (B, 1024)
    
    # Rename text keys
    if 'desc_embedding' in remapped:
        remapped['text_features'] = remapped.pop('desc_embedding')
    if 'desc_mask' in remapped:
        remapped['text_mask'] = remapped.pop('desc_mask')
    
    return remapped

# Test remapping and verify topology fields
test_batch = next(iter(train_loader))
test_batch = remap_batch(test_batch)
print("Remapped batch keys:")
for k, v in test_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

# Verify topology fields are present
assert 'edge_to_faces' in test_batch, "edge_to_faces required!"
assert 'bfs_level' in test_batch, "bfs_level required!"
print("\nTopology fields verified!")

In [ ]:
# Cell 7: Create Model (same architecture as v4.8.2)
gc.collect()
torch.cuda.empty_cache()

from clip4cad.models.clip4cad_gfa_v4_8_2 import CLIP4CAD_GFA_v482, GFAv482Config

# Model configuration (same as v4.8.2)
config = GFAv482Config(
    # Input dimensions
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    # Model dimensions
    d=320,
    d_proj=160,
    # Hierarchical codebook
    n_category=20,
    n_type_per_cat=10,
    n_variant_per_type=4,
    n_spatial=20,
    code_sparsity=0.1,
    # Architecture
    num_msg_layers=3,
    num_brep_tf_layers=5,
    num_heads=10,
    dropout=0.1,
)

model = CLIP4CAD_GFA_v482(config).to(device)

# Enable gradient checkpointing
model.enable_gradient_checkpointing()

print(f"\nModel: CLIP4CAD_GFA_v482")
print(f"Parameters: {model.count_parameters():,}")
print(f"Total codes: {model.codebook.total_codes}")

In [ ]:
# Cell 8: Load v4.8.3 Loss Function (FIXED cross-modal alignment)
import importlib
import clip4cad.losses.gfa_v4_8_3_losses as gfa_v483
importlib.reload(gfa_v483)

from clip4cad.losses.gfa_v4_8_3_losses import (
    GFAv483Loss,
    compute_cross_modal_metrics,
    compute_retrieval_metrics,
)

criterion = GFAv483Loss(
    tau=0.05,           # Lower temperature for sharper learning
    lambda_align=1.0,   # Strong bidirectional alignment
    lambda_code=0.2,    # Reduced code loss
    lambda_diversity=0.1,
    lambda_recon=0.1,
    label_smoothing=0.1,
)

# Scaler for mixed precision
scaler = GradScaler('cuda')

print("Loss function: GFAv483Loss (Fixed Cross-Modal)")
print(f"  tau: {criterion.tau}")
print(f"  lambda_align: {criterion.lambda_align}")
print(f"  lambda_code: {criterion.lambda_code}")
print(f"  label_smoothing: {criterion.label_smoothing}")
print("\nKey fixes:")
print("  - Bidirectional alignment (no detach on text)")
print("  - Text-anchored contrastive (2x weight)")
print("  - All modalities trained together from start")

In [ ]:
# Cell 9: Verify Forward Pass and Cross-Modal Metrics
print("Verifying forward pass with cross-modal metrics...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast('cuda'):
        outputs = model(test_batch, stage=1)  # Full forward with all modalities

# Check cross-modal alignment
metrics = compute_cross_modal_metrics(
    outputs['z_text'], outputs['z_brep'], outputs['z_pc']
)

print("\nCross-Modal Metrics (BEFORE training):")
print(f"  Text-BRep cosine: {metrics['cos_text_brep']:.4f}")
print(f"  Text-PC cosine:   {metrics['cos_text_pc']:.4f}")
print(f"  BRep-PC cosine:   {metrics['cos_brep_pc']:.4f}")
print(f"  Text-BRep gap:    {metrics['gap_text_brep']:.4f}")
print(f"  Text-PC gap:      {metrics['gap_text_pc']:.4f}")
print(f"  BRep-PC gap:      {metrics['gap_brep_pc']:.4f}")

# Quick retrieval check
ret = compute_retrieval_metrics(outputs['z_text'], outputs['z_brep'])
print(f"\nText->BRep Retrieval (batch): R@1={ret['R@1']:.1f}%, R@5={ret['R@5']:.1f}%")

# Test loss
loss, loss_dict = criterion(outputs, stage=0, epoch=1)
print("\nLoss components:")
for k, v in loss_dict.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.item():.4f}")

## Stage 0: Full 3-Way Alignment

**Key difference from v4.8.2:** All three modalities (Text, BRep, PC) train together from the start.

No separate BRep-PC pre-training that creates a closed space before text joins.

**Expected:** Text-BRep and Text-PC cosine should increase together, not just BRep-PC.

In [ ]:
# Cell 10: Stage 0 Training - Full 3-Way Alignment
print("\n" + "="*70)
print("STAGE 0: Full 3-Way Alignment (Text + BRep + PC together)")
print("="*70)

# Optimizer with warmup + cosine decay
optimizer = AdamW(model.parameters(), lr=STAGE0_LR, weight_decay=WEIGHT_DECAY)

# Warmup + Cosine scheduler
warmup_scheduler = LinearLR(
    optimizer, 
    start_factor=0.1, 
    end_factor=1.0, 
    total_iters=STAGE0_WARMUP * STEPS_PER_EPOCH
)
cosine_scheduler = CosineAnnealingLR(
    optimizer, 
    T_max=(STAGE0_EPOCHS - STAGE0_WARMUP) * STEPS_PER_EPOCH,
    eta_min=1e-6
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[STAGE0_WARMUP * STEPS_PER_EPOCH]
)

# History tracking
stage0_history = {
    'loss': [],
    'cos_text_brep': [], 'cos_text_pc': [], 'cos_brep_pc': [],
    'gap_text_brep': [], 'gap_text_pc': [], 'gap_brep_pc': [],
    'r1_text_brep': [], 'r1_brep_pc': [],
    'lr': [],
}

best_cos_tb = -1.0

for epoch in range(1, STAGE0_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'cos_tb': [], 'cos_tp': [], 'cos_bp': [],
        'gap_tb': [], 'gap_tp': [], 'gap_bp': [],
    }
    
    pbar = tqdm(train_loader, desc=f"Stage 0, Epoch {epoch}/{STAGE0_EPOCHS}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        optimizer.zero_grad()
        
        with autocast('cuda'):
            outputs = model(batch, stage=1)  # Full forward
            loss, loss_dict = criterion(outputs, stage=0, epoch=epoch)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss.item()
        
        # Track cross-modal metrics
        with torch.no_grad():
            metrics = compute_cross_modal_metrics(
                outputs['z_text'], outputs['z_brep'], outputs['z_pc']
            )
            epoch_metrics['cos_tb'].append(metrics['cos_text_brep'])
            epoch_metrics['cos_tp'].append(metrics['cos_text_pc'])
            epoch_metrics['cos_bp'].append(metrics['cos_brep_pc'])
            epoch_metrics['gap_tb'].append(metrics['gap_text_brep'])
        
        pbar.set_postfix({
            'loss': f"{loss.item():.3f}",
            'T-B': f"{metrics['cos_text_brep']:.3f}",  # TEXT-BREP!
            'T-P': f"{metrics['cos_text_pc']:.3f}",    # TEXT-PC!
            'B-P': f"{metrics['cos_brep_pc']:.3f}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_cos_tb = np.mean(epoch_metrics['cos_tb'])
    avg_cos_tp = np.mean(epoch_metrics['cos_tp'])
    avg_cos_bp = np.mean(epoch_metrics['cos_bp'])
    avg_gap_tb = np.mean(epoch_metrics['gap_tb'])
    current_lr = scheduler.get_last_lr()[0]
    
    stage0_history['loss'].append(avg_loss)
    stage0_history['cos_text_brep'].append(avg_cos_tb)
    stage0_history['cos_text_pc'].append(avg_cos_tp)
    stage0_history['cos_brep_pc'].append(avg_cos_bp)
    stage0_history['gap_text_brep'].append(avg_gap_tb)
    stage0_history['lr'].append(current_lr)
    
    # Quick retrieval check every 5 epochs
    r1_tb = 0.0
    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            val_batch = remap_batch(next(iter(val_loader)))
            with autocast('cuda'):
                val_out = model(val_batch, stage=1)
            ret = compute_retrieval_metrics(val_out['z_text'], val_out['z_brep'])
            r1_tb = ret['R@1']
        model.train()
    stage0_history['r1_text_brep'].append(r1_tb)
    
    # Save best model
    if avg_cos_tb > best_cos_tb:
        best_cos_tb = avg_cos_tb
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_cos_tb': best_cos_tb,
            'config': config,
        }, OUTPUT_DIR / 'checkpoint_best.pt')
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f} | "
          f"Text-BRep={avg_cos_tb:.4f} | Text-PC={avg_cos_tp:.4f} | BRep-PC={avg_cos_bp:.4f} | "
          f"Gap(T-B)={avg_gap_tb:.4f}" + (f" | R@1={r1_tb:.1f}%" if r1_tb > 0 else ""))

# Save Stage 0 checkpoint
torch.save({
    'epoch': STAGE0_EPOCHS,
    'model_state_dict': model.state_dict(),
    'history': stage0_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_stage0.pt')

print(f"\nStage 0 complete! Best Text-BRep cosine: {best_cos_tb:.4f}")

In [ ]:
# Cell 11: Mid-Training Evaluation
print("\nEvaluating after Stage 0...")

model.eval()
all_z_text = []
all_z_brep = []
all_z_pc = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast('cuda'):
            outputs = model(batch, stage=1)
        all_z_text.append(outputs['z_text'].cpu())
        all_z_brep.append(outputs['z_brep'].cpu())
        all_z_pc.append(outputs['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

# Cross-modal metrics
metrics = compute_cross_modal_metrics(z_text, z_brep, z_pc)
print("\nCross-Modal Alignment:")
print(f"  Text-BRep cosine: {metrics['cos_text_brep']:.4f}")
print(f"  Text-PC cosine:   {metrics['cos_text_pc']:.4f}")
print(f"  BRep-PC cosine:   {metrics['cos_brep_pc']:.4f}")

# Retrieval
ret_tb = compute_retrieval_metrics(z_text, z_brep)
ret_tp = compute_retrieval_metrics(z_text, z_pc)
ret_bp = compute_retrieval_metrics(z_brep, z_pc)

print("\nRetrieval Results:")
print(f"  Text -> BRep: R@1={ret_tb['R@1']:.1f}%, R@5={ret_tb['R@5']:.1f}%, R@10={ret_tb['R@10']:.1f}%")
print(f"  Text -> PC:   R@1={ret_tp['R@1']:.1f}%, R@5={ret_tp['R@5']:.1f}%")
print(f"  BRep -> PC:   R@1={ret_bp['R@1']:.1f}%, R@5={ret_bp['R@5']:.1f}%")

## Stage 1: Hard Negative Refinement

Fine-tune with hard negatives to improve discrimination.

In [ ]:
# Cell 12: Mine Hard Negatives
print("\n" + "="*70)
print("Mining hard negatives...")
print("="*70)

def mine_hard_negatives_simple(model, loader, device, top_k=10, max_batches=50, remap_fn=None):
    """Simple hard negative mining based on embedding similarity."""
    model.eval()
    
    all_z_text = []
    all_z_brep = []
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(loader, desc="Collecting embeddings")):
            if i >= max_batches:
                break
            if remap_fn:
                batch = remap_fn(batch)
            with autocast('cuda'):
                outputs = model(batch, stage=1)
            all_z_text.append(F.normalize(outputs['z_text'], dim=-1).cpu())
            all_z_brep.append(F.normalize(outputs['z_brep'], dim=-1).cpu())
    
    z_text = torch.cat(all_z_text, dim=0)
    z_brep = torch.cat(all_z_brep, dim=0)
    
    # Find hard negatives (high similarity but not matching)
    sims = z_text @ z_brep.T
    N = sims.shape[0]
    
    # Mask out diagonal (true matches)
    mask = ~torch.eye(N, dtype=torch.bool)
    
    hard_negs = {}
    for i in range(N):
        row_sims = sims[i].clone()
        row_sims[i] = -float('inf')  # Exclude self
        _, top_indices = row_sims.topk(top_k)
        hard_negs[i] = top_indices.tolist()
    
    return hard_negs

hard_negatives = mine_hard_negatives_simple(
    model, train_loader, device,
    top_k=10, max_batches=50,
    remap_fn=remap_batch
)
print(f"Mined {len(hard_negatives)} hard negative sets")

In [ ]:
# Cell 13: Stage 1 Training - Hard Negative Refinement
print("\n" + "="*70)
print("STAGE 1: Hard Negative Refinement")
print("="*70)

# Lower LR for fine-tuning
optimizer = AdamW(model.parameters(), lr=STAGE1_LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=STAGE1_EPOCHS * STEPS_PER_EPOCH, eta_min=1e-7)

stage1_history = {
    'loss': [],
    'cos_text_brep': [], 'cos_text_pc': [], 'cos_brep_pc': [],
    'r1_text_brep': [],
}

for epoch in range(1, STAGE1_EPOCHS + 1):
    # Re-mine every 3 epochs
    if epoch > 1 and epoch % 3 == 0:
        print("Re-mining hard negatives...")
        hard_negatives = mine_hard_negatives_simple(
            model, train_loader, device,
            top_k=10, max_batches=50,
            remap_fn=remap_batch
        )
    
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'cos_tb': [], 'cos_tp': [], 'cos_bp': []}
    
    pbar = tqdm(train_loader, desc=f"Stage 1, Epoch {epoch}/{STAGE1_EPOCHS}")
    
    for batch_idx, batch in enumerate(pbar):
        batch = remap_batch(batch)
        optimizer.zero_grad()
        
        # Get hard negatives for this batch
        batch_size = batch['face_features'].shape[0]
        start_idx = batch_idx * batch_size
        batch_hard_negs = [
            hard_negatives.get(start_idx + i) for i in range(batch_size)
        ]
        
        with autocast('cuda'):
            outputs = model(batch, stage=1)
            loss, loss_dict = criterion(outputs, stage=1, epoch=epoch, hard_negatives=batch_hard_negs)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        epoch_loss += loss.item()
        
        with torch.no_grad():
            metrics = compute_cross_modal_metrics(
                outputs['z_text'], outputs['z_brep'], outputs['z_pc']
            )
            epoch_metrics['cos_tb'].append(metrics['cos_text_brep'])
            epoch_metrics['cos_tp'].append(metrics['cos_text_pc'])
            epoch_metrics['cos_bp'].append(metrics['cos_brep_pc'])
        
        pbar.set_postfix({
            'loss': f"{loss.item():.3f}",
            'T-B': f"{metrics['cos_text_brep']:.3f}",
            'T-P': f"{metrics['cos_text_pc']:.3f}",
        })
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_cos_tb = np.mean(epoch_metrics['cos_tb'])
    avg_cos_tp = np.mean(epoch_metrics['cos_tp'])
    avg_cos_bp = np.mean(epoch_metrics['cos_bp'])
    
    stage1_history['loss'].append(avg_loss)
    stage1_history['cos_text_brep'].append(avg_cos_tb)
    stage1_history['cos_text_pc'].append(avg_cos_tp)
    stage1_history['cos_brep_pc'].append(avg_cos_bp)
    
    # Quick retrieval check
    model.eval()
    with torch.no_grad():
        val_batch = remap_batch(next(iter(val_loader)))
        with autocast('cuda'):
            val_out = model(val_batch, stage=1)
        ret = compute_retrieval_metrics(val_out['z_text'], val_out['z_brep'])
        r1_tb = ret['R@1']
    stage1_history['r1_text_brep'].append(r1_tb)
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f} | "
          f"Text-BRep={avg_cos_tb:.4f} | Text-PC={avg_cos_tp:.4f} | BRep-PC={avg_cos_bp:.4f} | "
          f"R@1={r1_tb:.1f}%")

# Save final checkpoint
torch.save({
    'epoch': TOTAL_EPOCHS,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'config': config,
}, OUTPUT_DIR / 'checkpoint_final.pt')

print("\nStage 1 complete!")

In [ ]:
# Cell 14: Final Evaluation
print("\n" + "="*70)
print("FINAL EVALUATION")
print("="*70)

model.eval()
all_z_text = []
all_z_brep = []
all_z_pc = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast('cuda'):
            outputs = model(batch, stage=1)
        all_z_text.append(outputs['z_text'].cpu())
        all_z_brep.append(outputs['z_brep'].cpu())
        all_z_pc.append(outputs['z_pc'].cpu())

z_text = torch.cat(all_z_text, dim=0)
z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

N = z_text.shape[0]
print(f"\nEvaluation set: {N} samples")

# Cross-modal metrics
metrics = compute_cross_modal_metrics(z_text, z_brep, z_pc)
print("\nCross-Modal Alignment:")
print(f"  Text-BRep cosine: {metrics['cos_text_brep']:.4f}")
print(f"  Text-PC cosine:   {metrics['cos_text_pc']:.4f}")
print(f"  BRep-PC cosine:   {metrics['cos_brep_pc']:.4f}")
print(f"  Text-BRep gap:    {metrics['gap_text_brep']:.4f}")
print(f"  Text-PC gap:      {metrics['gap_text_pc']:.4f}")
print(f"  BRep-PC gap:      {metrics['gap_brep_pc']:.4f}")

# Retrieval
ret_tb = compute_retrieval_metrics(z_text, z_brep)
ret_tp = compute_retrieval_metrics(z_text, z_pc)
ret_bp = compute_retrieval_metrics(z_brep, z_pc)

print("\n" + "="*50)
print("Retrieval Results (v4.8.3):")
print("="*50)
print(f"Text -> BRep: R@1={ret_tb['R@1']:.1f}%, R@5={ret_tb['R@5']:.1f}%, R@10={ret_tb['R@10']:.1f}%")
print(f"Text -> PC:   R@1={ret_tp['R@1']:.1f}%, R@5={ret_tp['R@5']:.1f}%, R@10={ret_tp['R@10']:.1f}%")
print(f"BRep -> PC:   R@1={ret_bp['R@1']:.1f}%, R@5={ret_bp['R@5']:.1f}%, R@10={ret_bp['R@10']:.1f}%")
print("="*50)

In [ ]:
# Cell 15: Training Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Cross-Modal Cosine Similarity
ax = axes[0, 0]
s0_epochs = list(range(1, len(stage0_history['cos_text_brep']) + 1))
s1_epochs = list(range(len(s0_epochs) + 1, len(s0_epochs) + len(stage1_history['cos_text_brep']) + 1))

ax.plot(s0_epochs, stage0_history['cos_text_brep'], 'b-', label='Text-BRep (Stage 0)')
ax.plot(s0_epochs, stage0_history['cos_text_pc'], 'g-', label='Text-PC (Stage 0)')
ax.plot(s0_epochs, stage0_history['cos_brep_pc'], 'r-', label='BRep-PC (Stage 0)')
if stage1_history['cos_text_brep']:
    ax.plot(s1_epochs, stage1_history['cos_text_brep'], 'b--', label='Text-BRep (Stage 1)')
    ax.plot(s1_epochs, stage1_history['cos_text_pc'], 'g--', label='Text-PC (Stage 1)')
    ax.plot(s1_epochs, stage1_history['cos_brep_pc'], 'r--', label='BRep-PC (Stage 1)')
ax.axhline(y=0.5, color='k', linestyle=':', alpha=0.3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Cross-Modal Alignment')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Text-BRep Gap
ax = axes[0, 1]
ax.plot(s0_epochs, stage0_history['gap_text_brep'], 'b-', label='Stage 0')
ax.axhline(y=0, color='g', linestyle='--', alpha=0.5, label='Target (0)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Gap (L2)')
ax.set_title('Text-BRep Modality Gap')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[0, 2]
all_loss = stage0_history['loss'] + stage1_history['loss']
ax.plot(all_loss)
ax.axvline(x=STAGE0_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 1 start')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning Rate
ax = axes[1, 0]
ax.plot(stage0_history['lr'])
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Stage 0 Learning Rate')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Retrieval R@1
ax = axes[1, 1]
r1_vals = [r for r in stage0_history['r1_text_brep'] if r > 0]
r1_epochs = [i+1 for i, r in enumerate(stage0_history['r1_text_brep']) if r > 0]
if r1_vals:
    ax.plot(r1_epochs, r1_vals, 'bo-', label='Stage 0')
if stage1_history['r1_text_brep']:
    ax.plot(s1_epochs, stage1_history['r1_text_brep'], 'go-', label='Stage 1')
ax.set_xlabel('Epoch')
ax.set_ylabel('R@1 (%)')
ax.set_title('Text->BRep Retrieval R@1')
ax.legend()
ax.grid(True, alpha=0.3)

# Summary
ax = axes[1, 2]
ax.axis('off')
summary_text = (
    f"GFA v4.8.3 Training Summary\n"
    f"(Fixed Cross-Modal Alignment)\n\n"
    f"Model: d={config.d}, {model.count_parameters():,} params\n"
    f"Codes: {model.codebook.total_codes}\n\n"
    f"Stage 0 (3-Way): {STAGE0_EPOCHS} epochs\n"
    f"  Text-BRep cos: {stage0_history['cos_text_brep'][-1]:.4f}\n"
    f"  Text-PC cos:   {stage0_history['cos_text_pc'][-1]:.4f}\n"
    f"  BRep-PC cos:   {stage0_history['cos_brep_pc'][-1]:.4f}\n\n"
    f"Final Retrieval (Text->BRep):\n"
    f"  R@1: {ret_tb['R@1']:.1f}%\n"
    f"  R@5: {ret_tb['R@5']:.1f}%\n"
    f"  R@10: {ret_tb['R@10']:.1f}%"
)
ax.text(0.5, 0.5, summary_text, ha='center', va='center',
        fontsize=10, family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves_v483.png', dpi=150)
plt.show()

In [ ]:
# Cell 16: Save Final Model
torch.save({
    'config': config,
    'model_state_dict': model.state_dict(),
    'stage0_history': stage0_history,
    'stage1_history': stage1_history,
    'final_metrics': {
        'cos_text_brep': metrics['cos_text_brep'],
        'cos_text_pc': metrics['cos_text_pc'],
        'cos_brep_pc': metrics['cos_brep_pc'],
        'r1_text_brep': ret_tb['R@1'],
        'r5_text_brep': ret_tb['R@5'],
        'r10_text_brep': ret_tb['R@10'],
    },
}, OUTPUT_DIR / 'gfa_v4_8_3_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'gfa_v4_8_3_final.pt'}")
print(f"\nFinal Results:")
print(f"  Text-BRep cosine: {metrics['cos_text_brep']:.4f}")
print(f"  Text->BRep R@1: {ret_tb['R@1']:.1f}%")
print(f"  Text->BRep R@5: {ret_tb['R@5']:.1f}%")